# ScholarLens 00 - Setup check and data exploration
Owner: whole team (run first). Verifies the code works, downloads all datasets and produces the Datasets-section statistics and figures for the report.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sajithaucm/Scholarlens_Project/blob/main/notebooks/00_setup_and_eda.ipynb)

## 0. Setup (run every time you open this notebook)
1. **Runtime -> Change runtime type -> T4 GPU** (Hardware accelerator).
2. Replace `sajithaucm` below with your team's GitHub account (once, then commit).
3. Checkpoints are saved to **Google Drive** so they survive a disconnect; small result files are saved in the repo's `results/` folder and pushed to GitHub at the end.

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "https://github.com/sajithaucm/Scholarlens_Project.git"
if not os.path.exists("/content/Scholarlens_Project"):
    !git clone $REPO /content/Scholarlens_Project
%cd /content/Scholarlens_Project
!git pull -q
!pip install -q -r requirements.txt

# Checkpoints -> Google Drive (big files never go to GitHub)
os.environ["SCHOLARLENS_OUTPUT_DIR"] = "/content/drive/MyDrive/Scholarlens_Project/outputs"
os.makedirs(os.environ["SCHOLARLENS_OUTPUT_DIR"], exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip freeze > results/environment.txt   # exact library versions, for reproducibility

## 1. Run the test suite
Tiny models on synthetic data: every training script, the pipeline and the demo must pass **before** spending GPU time.

In [ ]:
!python -m pytest tests/ -q -p no:warnings

## 2. Load the three datasets

In [ ]:
from src.utils import load_config
from src.data_loading import load_summarization, load_qa, load_ner
from src import eda
cfg = load_config()
summ = load_summarization(cfg)
qa = load_qa(cfg)
ner = load_ner(cfg)
print(summ, qa, ner, sep='\n')

## 3. Dataset statistics (for the report's *Datasets* section)
The rubric requires **>= 50,000 sentences**. `est_total_sentences` shows the arXiv training subset alone is far above that.

In [ ]:
s = eda.to_frame(eda.summarization_stats(summ)); s

In [ ]:
q = eda.to_frame(eda.qa_stats(qa)); q

In [ ]:
n = eda.to_frame(eda.ner_stats(ner)); n

In [ ]:
import pandas as pd
with open('results/dataset_stats.md', 'w') as f:
    for name, df in (('Summarization (arXiv)', s), ('QA (SQuAD 2.0)', q), ('NER (SciERC)', n)):
        f.write(f'## {name}\n\n' + df.to_markdown() + '\n\n')
print(open('results/dataset_stats.md').read())

## 4. Figures

In [ ]:
from pathlib import Path
from IPython.display import Image, display
tr = summ['train']
eda.plot_length_hist([len(a.split()) for a in tr['article']], [len(a.split()) for a in tr['abstract']],
                     'Article length', 'Abstract length', 'arXiv training set', Path('results/fig_eda_arxiv_lengths.png'))
ctx = list(set(qa['train']['context']))
eda.plot_length_hist([len(c.split()) for c in ctx], [len(qq.split()) for qq in qa['train']['question']],
                     'Context length', 'Question length', 'SQuAD 2.0 training subset', Path('results/fig_eda_squad_lengths.png'))
eda.plot_entity_counts(eda.ner_stats(ner), Path('results/fig_eda_scierc_entities.png'))
for f in ('fig_eda_arxiv_lengths', 'fig_eda_squad_lengths', 'fig_eda_scierc_entities'):
    display(Image(f'results/{f}.png'))

## 5. Preprocessing example (before / after cleaning) - screenshot for the report

In [ ]:
from datasets import load_dataset
from src.preprocessing import clean_text
raw = load_dataset(cfg['summarization']['dataset'], split='test[:1]')[0]['article'][:800]
print('BEFORE:\n', raw, '\n\nAFTER:\n', clean_text(raw))

## Save results to GitHub
Results (`results/*.json`, figures, sample predictions) are small and must be committed.
Create a GitHub **fine-grained personal access token** (repo contents: read & write) and paste it when asked. It is not stored in the notebook.
Alternative: download the `results/` folder (Files panel) and commit it from VS Code.

In [ ]:
from getpass import getpass
name, email = input("Your name: "), input("Your GitHub email: ")
token = getpass("GitHub token: ")
!git config user.name "$name" && git config user.email "$email"
!git add results/
!git commit -m "Add results from Colab run" || echo "nothing to commit"
!git push https://$token@github.com/sajithaucm/Scholarlens_Project.git HEAD:main
del token